In [1]:
#%pip install ipywidgets matplotlib pillow

In [2]:
from pathlib import Path
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
import re

# --------------------------------------------------
# 1. Directories containing the fitted spectra
# --------------------------------------------------

DIR = Path(
    r"C:\Users\javas\Dropbox\GitHub\VIENTO\structure_function_analysis\FLAMES_30dor\gausspyp"
)

folders = {
    "1 component": DIR / "gausspyp_v02.1" / "gausspy_validation" / "plots_validation",
    "2 components": DIR / "gausspyp_v02.2" / "gausspy_validation" / "plots_validation",
    "5 components": DIR / "gausspyp_v02.3" / "gausspy_validation" / "plots_validation",
}

extensions = {".png", ".jpg", ".jpeg"}

# --------------------------------------------------
# 2. Read image filenames
# --------------------------------------------------

catalogs = {}

for label, folder in folders.items():

    catalogs[label] = {
        p.stem: p
        for p in folder.iterdir()
        if p.suffix.lower() in extensions
    }

# Only include spectra available in all folders
common = set.intersection(
    *(set(files) for files in catalogs.values())
)

def natural_key(name):
    return [
        int(x) if x.isdigit() else x.lower()
        for x in re.split(r"(\d+)", name)
    ]

spectrum_ids = sorted(common, key=natural_key)

if not spectrum_ids:
    raise ValueError(
        "No matching image filenames found across the folders."
    )

print(f"Found {len(spectrum_ids)} spectra for comparison.")

# --------------------------------------------------
# 3. Create widgets
# --------------------------------------------------

selector = widgets.Dropdown(
    options=spectrum_ids,
    description="Spectrum:",
    layout=widgets.Layout(width="350px")
)

previous = widgets.Button(description="← Previous")
following = widgets.Button(description="Next →")

output = widgets.Output()

# --------------------------------------------------
# 4. Display the three fits
# --------------------------------------------------

def show_spectrum(change=None):

    spectrum_id = selector.value

    with output:
        clear_output(wait=True)

        fig, axes = plt.subplots(
            1, 3,
            figsize=(18, 5),
            constrained_layout=True
        )

        for ax, (label, files) in zip(
            axes, catalogs.items()
        ):

            with Image.open(files[spectrum_id]) as img:
                ax.imshow(img.copy())

            ax.set_title(label)
            ax.axis("off")

        fig.suptitle(
            f"Spectrum {spectrum_id}",
            fontsize=15
        )

        display(fig)
        plt.close(fig)

# --------------------------------------------------
# 5. Navigation buttons
# --------------------------------------------------

def navigate(step):

    index = spectrum_ids.index(selector.value)

    new_index = max(
        0,
        min(index + step, len(spectrum_ids) - 1)
    )

    selector.value = spectrum_ids[new_index]

previous.on_click(lambda b: navigate(-1))
following.on_click(lambda b: navigate(1))

selector.observe(show_spectrum, names="value")

# --------------------------------------------------
# 6. Display interface
# --------------------------------------------------

display(
    widgets.VBox([
        widgets.HBox([previous, selector, following]),
        output
    ])
)

show_spectrum()

Found 1382 spectra for comparison.


In [3]:
import ipywidgets as widgets
from IPython.display import display

button = widgets.Button(description="Click me")
status = widgets.HTML(value="Button not clicked")

def test_click(b):
    status.value = "<b>Button works!</b>"

button.on_click(test_click)

display(widgets.VBox([button, status]))